# Predicting Student Test Scores
## Score: 8.71688

In [1]:
import subprocess
import sys

try:
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    import xgboost as xgb
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge, Lasso, ElasticNet
    from sklearn.cluster import KMeans
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "catboost", "xgboost", "--quiet"])
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    import xgboost as xgb
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge, Lasso, ElasticNet
    from sklearn.cluster import KMeans

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')
test_ids = test['id'].copy()


In [3]:
def create_features(df, train_df=None):
    df = df.copy()
    
    df['internet_access'] = (df['internet_access'] == 'yes').astype(int)
    
    sleep_quality_map = {'poor': 0, 'average': 1, 'good': 2}
    facility_rating_map = {'low': 0, 'medium': 1, 'high': 2}
    exam_difficulty_map = {'easy': 0, 'moderate': 1, 'hard': 2}
    
    df['sleep_quality_ord'] = df['sleep_quality'].map(sleep_quality_map)
    df['facility_rating_ord'] = df['facility_rating'].map(facility_rating_map)
    df['exam_difficulty_ord'] = df['exam_difficulty'].map(exam_difficulty_map)
    
    df['study_efficiency'] = df['study_hours'] * df['class_attendance']
    df['study_sleep_ratio'] = df['study_hours'] / (df['sleep_hours'] + 1e-5)
    df['attendance_facility'] = df['class_attendance'] * df['facility_rating_ord']
    df['sleep_quality_score'] = df['sleep_hours'] * df['sleep_quality_ord']
    df['study_hours_sq'] = df['study_hours'] ** 2
    df['attendance_sq'] = df['class_attendance'] ** 2
    df['study_per_age'] = df['study_hours'] / (df['age'] + 1e-5)
    df['attendance_per_age'] = df['class_attendance'] / (df['age'] + 1e-5)
    df['study_per_attendance'] = df['study_hours'] / (df['class_attendance'] + 1e-5)
    df['sleep_per_age'] = df['sleep_hours'] / (df['age'] + 1e-5)
    
    df['study_hours_cubed'] = df['study_hours'] ** 3
    df['study_attendance_sleep'] = df['study_hours'] * df['class_attendance'] * df['sleep_hours']
    df['efficiency_sleep'] = df['study_efficiency'] * df['sleep_hours']
    df['study_facility'] = df['study_hours'] * df['facility_rating_ord']
    df['difficulty_facility'] = df['exam_difficulty_ord'] * df['facility_rating_ord']
    df['study_attendance_facility'] = df['study_hours'] * df['class_attendance'] * df['facility_rating_ord']
    df['sleep_attendance'] = df['sleep_hours'] * df['class_attendance']
    df['study_difficulty'] = df['study_hours'] * df['exam_difficulty_ord']
    df['attendance_difficulty'] = df['class_attendance'] * df['exam_difficulty_ord']
    df['total_effort'] = df['study_hours'] + df['class_attendance'] / 10
    df['sleep_ratio_sq'] = df['study_sleep_ratio'] ** 2
    df['study_attendance_ratio'] = df['study_hours'] / (df['class_attendance'] + 1e-5)
    df['efficiency_per_sleep'] = df['study_efficiency'] / (df['sleep_hours'] + 1e-5)
    df['study_facility_difficulty'] = df['study_hours'] * df['facility_rating_ord'] * df['exam_difficulty_ord']
    df['attendance_sleep_quality'] = df['class_attendance'] * df['sleep_hours'] * df['sleep_quality_ord']
    
    # More interactions with ordinal features
    df['study_sleep_quality'] = df['study_hours'] * df['sleep_quality_ord']
    df['attendance_sleep_quality_ord'] = df['class_attendance'] * df['sleep_quality_ord']
    df['study_facility_sleep_quality'] = df['study_hours'] * df['facility_rating_ord'] * df['sleep_quality_ord']
    df['attendance_difficulty_sleep'] = df['class_attendance'] * df['exam_difficulty_ord'] * df['sleep_hours']
    df['efficiency_difficulty'] = df['study_efficiency'] * df['exam_difficulty_ord']
    df['efficiency_facility'] = df['study_efficiency'] * df['facility_rating_ord']
    df['study_hours_log'] = np.log1p(df['study_hours'])
    df['attendance_log'] = np.log1p(df['class_attendance'])
    df['sleep_hours_log'] = np.log1p(df['sleep_hours'])
    df['age_squared'] = df['age'] ** 2
    
    df['study_hours_bin'] = pd.cut(df['study_hours'], bins=5, labels=False, duplicates='drop')
    df['attendance_bin'] = pd.cut(df['class_attendance'], bins=5, labels=False, duplicates='drop')
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 20, 22, 25], labels=[0, 1, 2, 3], duplicates='drop')
    df['age_group'] = df['age_group'].fillna(2).astype(int)
    
    if train_df is not None:
        top_features_for_interactions = ['study_hours', 'class_attendance', 'sleep_hours', 'study_efficiency', 'study_sleep_ratio']
        for i, feat1 in enumerate(top_features_for_interactions):
            for feat2 in top_features_for_interactions[i+1:]:
                if feat1 in df.columns and feat2 in df.columns:
                    df[f'{feat1}_x_{feat2}'] = df[feat1] * df[feat2]
                    df[f'{feat1}_div_{feat2}'] = df[feat1] / (df[feat2] + 1e-5)
        
        # High-value 3-way interactions for top features
        top3_features = ['study_hours', 'class_attendance', 'sleep_hours']
        if all(feat in df.columns for feat in top3_features):
            df['study_attendance_sleep_3way'] = df['study_hours'] * df['class_attendance'] * df['sleep_hours']
        
        # Additional high-value 3-way interactions
        if 'study_efficiency' in df.columns and 'sleep_hours' in df.columns and 'facility_rating_ord' in df.columns:
            df['efficiency_sleep_facility_3way'] = df['study_efficiency'] * df['sleep_hours'] * df['facility_rating_ord']
        
        if 'study_hours' in df.columns and 'study_sleep_ratio' in df.columns and 'facility_rating_ord' in df.columns:
            df['study_ratio_facility_3way'] = df['study_hours'] * df['study_sleep_ratio'] * df['facility_rating_ord']
        numeric_cols = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
        cat_cols = ['course', 'study_method', 'gender']
        
        for cat_col in cat_cols:
            for num_col in numeric_cols:
                stats = train_df.groupby(cat_col)[num_col].agg(['mean', 'std', 'min', 'max', 'median'])
                df[f'{num_col}_mean_by_{cat_col}'] = df[cat_col].map(stats['mean'])
                df[f'{num_col}_std_by_{cat_col}'] = df[cat_col].map(stats['std'])
                df[f'{num_col}_min_by_{cat_col}'] = df[cat_col].map(stats['min'])
                df[f'{num_col}_max_by_{cat_col}'] = df[cat_col].map(stats['max'])
                df[f'{num_col}_median_by_{cat_col}'] = df[cat_col].map(stats['median'])
                df[f'{num_col}_diff_from_mean_{cat_col}'] = df[num_col] - df[f'{num_col}_mean_by_{cat_col}']
                df[f'{num_col}_diff_from_median_{cat_col}'] = df[num_col] - df[f'{num_col}_median_by_{cat_col}']
                
                # Add quantiles
                q25 = train_df.groupby(cat_col)[num_col].quantile(0.25)
                q75 = train_df.groupby(cat_col)[num_col].quantile(0.75)
                df[f'{num_col}_q25_by_{cat_col}'] = df[cat_col].map(q25)
                df[f'{num_col}_q75_by_{cat_col}'] = df[cat_col].map(q75)
        
        cluster_features = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
        kmeans.fit(train_df[cluster_features])
        df['cluster'] = kmeans.predict(df[cluster_features])
        df['cluster_dist_0'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[0], axis=1)
        df['cluster_dist_1'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[1], axis=1)
        df['cluster_dist_2'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[2], axis=1)
    
    return df

train = create_features(train, train_df=train)
test = create_features(test, train_df=train)

np.random.seed(42)
numeric_cols_for_aug = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
augmented_rows = []

# Increased augmentation: 7% Gaussian noise
for _ in range(int(len(train) * 0.07)):
    idx = np.random.randint(0, len(train))
    row = train.iloc[idx].copy()
    
    for col in numeric_cols_for_aug:
        if col in row.index:
            noise = np.random.normal(0, row[col] * 0.01)
            row[col] = max(0, row[col] + noise)
    
    augmented_rows.append(row)

if augmented_rows:
    train_aug = pd.DataFrame(augmented_rows)
    train = pd.concat([train, train_aug], ignore_index=True)


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_26292\2675537153.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{num_col}_median_by_{cat_col}'] = df[cat_col].map(stats['median'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_26292\2675537153.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{num_col}_diff_from_mean_{cat_col}'] = df[num_col] - df[f'{num_col}_mean_by_{cat_col}']
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_26292\2675537153.py:89: PerformanceWarning: DataFrame is highly fragmente

In [4]:
categorical_cols = ['gender', 'course', 'sleep_quality', 'study_method']
target_col = 'exam_score'

for col in categorical_cols:
    train[f'{col}_freq'] = train.groupby(col)[col].transform('count')
    test[f'{col}_freq'] = test[col].map(train.groupby(col)[col].count())

kf = KFold(n_splits=4, shuffle=True, random_state=42)

global_mean = train[target_col].mean()
smoothing = 8.0

for col in categorical_cols:
    train[f'{col}_target'] = 0.0
    test[f'{col}_target'] = 0.0
    train[f'{col}_target_std'] = 0.0
    test[f'{col}_target_std'] = 0.0
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]
        
        mean_target = train_fold.groupby(col)[target_col].mean()
        std_target = train_fold.groupby(col)[target_col].std()
        count = train_fold.groupby(col)[target_col].count()
        
        adaptive_smoothing = smoothing * (1 + 0.5 / (count + 1))
        smoothed = (mean_target * count + global_mean * adaptive_smoothing) / (count + adaptive_smoothing)
        
        train.loc[val_idx, f'{col}_target'] = val_fold[col].map(smoothed).astype(float)
        train.loc[val_idx, f'{col}_target_std'] = val_fold[col].map(std_target).fillna(0).astype(float)
        
        test_mean = test[col].map(train_fold.groupby(col)[target_col].mean())
        test_std = test[col].map(train_fold.groupby(col)[target_col].std())
        test_count = test[col].map(train_fold.groupby(col)[target_col].count()).fillna(0)
        test_smoothed = (test_mean * test_count + global_mean * adaptive_smoothing.mean()) / (test_count + adaptive_smoothing.mean())
        test[f'{col}_target'] += test_smoothed.fillna(global_mean).astype(float) / 4
        test[f'{col}_target_std'] += test_std.fillna(0).astype(float) / 4

# Target encoding for feature combinations
categorical_combinations = [
    ('course', 'study_method', 'course_study_method'),
    ('course', 'exam_difficulty', 'course_exam_difficulty'),
    ('course', 'facility_rating', 'course_facility_rating'),
    ('gender', 'course', 'gender_course'),
    ('study_method', 'exam_difficulty', 'study_method_exam_difficulty'),
    ('course', 'sleep_quality', 'course_sleep_quality')
]

for col1, col2, combo_name in categorical_combinations:
    train[combo_name] = train[col1].astype(str) + '_' + train[col2].astype(str)
    test[combo_name] = test[col1].astype(str) + '_' + test[col2].astype(str)
    
    train[f'{combo_name}_target'] = 0.0
    test[f'{combo_name}_target'] = 0.0
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]
        
        mean_target = train_fold.groupby(combo_name)[target_col].mean()
        count = train_fold.groupby(combo_name)[target_col].count()
        
        adaptive_smoothing = smoothing * (1 + 0.5 / (count + 1))
        smoothed = (mean_target * count + global_mean * adaptive_smoothing) / (count + adaptive_smoothing)
        
        train.loc[val_idx, f'{combo_name}_target'] = val_fold[combo_name].map(smoothed).fillna(global_mean).astype(float)
        
        test_mean = test[combo_name].map(train_fold.groupby(combo_name)[target_col].mean())
        test_count = test[combo_name].map(train_fold.groupby(combo_name)[target_col].count()).fillna(0)
        test_smoothed = (test_mean * test_count + global_mean * adaptive_smoothing.mean()) / (test_count + adaptive_smoothing.mean())
        test[f'{combo_name}_target'] += test_smoothed.fillna(global_mean).astype(float) / 4

numeric_cols_to_cap = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
for col in numeric_cols_to_cap:
    q1 = train[col].quantile(0.01)
    q99 = train[col].quantile(0.99)
    train[col] = train[col].clip(lower=q1, upper=q99)
    test[col] = test[col].clip(lower=q1, upper=q99)

# Collect all combination column names for dropping
combo_cols = [combo[2] for combo in categorical_combinations]
drop_cols = ['id', 'exam_score', 'gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty'] + combo_cols
X = train.drop(drop_cols, axis=1)
y = train[target_col]
X_test = test.drop(['id', 'gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty'], axis=1)

temp_kf = KFold(n_splits=3, shuffle=True, random_state=42)
feature_importance_scores = np.zeros(len(X.columns))
feature_names = X.columns.tolist()

for train_idx, val_idx in temp_kf.split(X):
    X_temp_train, X_temp_val = X.iloc[train_idx], X.iloc[val_idx]
    y_temp_train, y_temp_val = y.iloc[train_idx], y.iloc[val_idx]
    
    temp_model = xgb.XGBRegressor(n_estimators=250, learning_rate=0.1, max_depth=6, 
                                   random_state=42, tree_method='hist', subsample=0.8, colsample_bytree=0.8)
    temp_model.fit(X_temp_train, y_temp_train, 
                   eval_set=[(X_temp_val, y_temp_val)], verbose=False)
    feature_importance_scores += temp_model.feature_importances_ / 3

feature_importance = pd.DataFrame({'feature': feature_names, 'importance': feature_importance_scores})
feature_importance = feature_importance.sort_values('importance', ascending=False)
keep_pct = 0.94  # Increased from 0.92 to retain more potentially useful features
n_keep = int(len(feature_importance) * keep_pct)
selected_features = feature_importance.head(n_keep)['feature'].values
X = X[selected_features]
X_test = X_test[selected_features]


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_26292\384190421.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f'{col}_freq'] = train.groupby(col)[col].transform('count')
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_26292\384190421.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[f'{col}_freq'] = test[col].map(train.groupby(col)[col].count())
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_26292\384190421.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of ca

In [5]:
kf = KFold(n_splits=4, shuffle=True, random_state=42)


In [6]:
seeds = [42, 123, 456, 789, 999]  # Added more seeds for increased diversity

# Model-specific hyperparameters optimized for each model type
xgb_params_base = {
    'n_estimators': 1800,
    'learning_rate': 0.011,
    'max_depth': 10,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.10,
    'reg_lambda': 0.10,
    'tree_method': 'hist',
    'device': 'cpu'
}

lgb_params_base = {
    'n_estimators': 1800,
    'learning_rate': 0.011,
    'max_depth': 9,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.08,
    'reg_lambda': 0.08,
    'verbose': -1
}

cat_params_base = {
    'iterations': 1800,
    'learning_rate': 0.011,
    'depth': 9,
    'subsample': 0.80,
    'colsample_bylevel': 0.80,
    'l2_leaf_reg': 0.10,
    'verbose': False
}


In [7]:
all_oof = []
all_test_preds = []
model_names = []

for seed in seeds:
    xgb_params = {**xgb_params_base, 'random_state': seed}
    lgb_params = {**lgb_params_base, 'random_state': seed}
    cat_params = {**cat_params_base, 'random_seed': seed}
    
    xgb_oof = np.zeros(len(train))
    lgb_oof = np.zeros(len(train))
    cat_oof = np.zeros(len(train))
    hgb_oof = np.zeros(len(train))
    
    xgb_test_preds = np.zeros(len(test))
    lgb_test_preds = np.zeros(len(test))
    cat_test_preds = np.zeros(len(test))
    hgb_test_preds = np.zeros(len(test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_train_fold, y_train_fold, 
                      eval_set=[(X_val_fold, y_val_fold)],
                      verbose=False)
        xgb_oof[val_idx] = xgb_model.predict(X_val_fold)
        xgb_test_preds += xgb_model.predict(X_test) / 4
        
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(X_train_fold, y_train_fold,
                      eval_set=[(X_val_fold, y_val_fold)],
                      callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])
        lgb_oof[val_idx] = lgb_model.predict(X_val_fold)
        lgb_test_preds += lgb_model.predict(X_test) / 4
        
        cat_model = CatBoostRegressor(**cat_params)
        cat_model.fit(X_train_fold, y_train_fold,
                      eval_set=(X_val_fold, y_val_fold),
                      early_stopping_rounds=120)
        cat_oof[val_idx] = cat_model.predict(X_val_fold)
        cat_test_preds += cat_model.predict(X_test) / 4
        
        # Add HistGradientBoostingRegressor
        hgb_model = HistGradientBoostingRegressor(
            max_iter=1500, learning_rate=0.01, max_depth=9,
            random_state=seed, l2_regularization=0.1
        )
        hgb_model.fit(X_train_fold, y_train_fold)
        hgb_oof[val_idx] = hgb_model.predict(X_val_fold)
        hgb_test_preds += hgb_model.predict(X_test) / 4
    
    all_oof.append(xgb_oof)
    all_oof.append(lgb_oof)
    all_oof.append(cat_oof)
    all_oof.append(hgb_oof)
    all_test_preds.append(xgb_test_preds)
    all_test_preds.append(lgb_test_preds)
    all_test_preds.append(cat_test_preds)
    all_test_preds.append(hgb_test_preds)
    model_names.extend([f'xgb_{seed}', f'lgb_{seed}', f'cat_{seed}', f'hgb_{seed}'])
    
    xgb_rmse = np.sqrt(mean_squared_error(y, xgb_oof))
    lgb_rmse = np.sqrt(mean_squared_error(y, lgb_oof))
    cat_rmse = np.sqrt(mean_squared_error(y, cat_oof))
    hgb_rmse = np.sqrt(mean_squared_error(y, hgb_oof))
    print(f'Seed {seed} - XGB: {xgb_rmse:.5f}, LGB: {lgb_rmse:.5f}, Cat: {cat_rmse:.5f}, HGB: {hgb_rmse:.5f}')

all_oof = np.column_stack(all_oof)
all_test_preds = np.column_stack(all_test_preds)


Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[1800]	valid_0's l2: 77.1606
Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[1800]	valid_0's l2: 77.4537
Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[1800]	valid_0's l2: 77.2956
Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[1800]	valid_0's l2: 77.4979
Seed 42 - XGB: 8.68000, LGB: 8.79500, Cat: 8.80150, HGB: 8.81214
Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[1800]	valid_0's l2: 77.209
Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[1800]	valid_0's l2: 77.5348
Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iterat

In [8]:
stacking_oof = np.zeros(len(train))
stacking_test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_stack_train = all_oof[train_idx]
    X_stack_val = all_oof[val_idx]
    y_stack_train = y.iloc[train_idx]
    
    # Scale features for linear meta-learners
    scaler = StandardScaler()
    X_stack_train_scaled = scaler.fit_transform(X_stack_train)
    X_stack_val_scaled = scaler.transform(X_stack_val)
    
    ridge = Ridge(alpha=2.5, random_state=42)
    lasso = Lasso(alpha=0.25, random_state=42, max_iter=3000)
    elastic = ElasticNet(alpha=0.4, l1_ratio=0.6, random_state=42, max_iter=3000)
    xgb_meta = xgb.XGBRegressor(n_estimators=600, learning_rate=0.025, max_depth=6, 
                                random_state=42, tree_method='hist', subsample=0.85, colsample_bytree=0.85)
    lgb_meta = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.025, max_depth=6,
                                random_state=42, verbose=-1, subsample=0.85, colsample_bytree=0.85)
    cat_meta = CatBoostRegressor(iterations=600, learning_rate=0.025, depth=6,
                                random_seed=42, subsample=0.85, colsample_bylevel=0.85, verbose=False)
    
    # Use scaled features for linear models, original for tree models
    ridge.fit(X_stack_train_scaled, y_stack_train)
    lasso.fit(X_stack_train_scaled, y_stack_train)
    elastic.fit(X_stack_train_scaled, y_stack_train)
    xgb_meta.fit(X_stack_train, y_stack_train, 
                 eval_set=[(X_stack_val, y.iloc[val_idx])], verbose=False)
    lgb_meta.fit(X_stack_train, y_stack_train,
                 eval_set=[(X_stack_val, y.iloc[val_idx])],
                 callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])
    cat_meta.fit(X_stack_train, y_stack_train,
                 eval_set=(X_stack_val, y.iloc[val_idx]),
                 early_stopping_rounds=120)
    
    ridge_pred = ridge.predict(X_stack_val_scaled)
    lasso_pred = lasso.predict(X_stack_val_scaled)
    elastic_pred = elastic.predict(X_stack_val_scaled)
    xgb_meta_pred = xgb_meta.predict(X_stack_val)
    lgb_meta_pred = lgb_meta.predict(X_stack_val)
    cat_meta_pred = cat_meta.predict(X_stack_val)
    
    meta_preds = np.column_stack([ridge_pred, lasso_pred, elastic_pred, xgb_meta_pred, lgb_meta_pred, cat_meta_pred])
    meta_rmses = [np.sqrt(mean_squared_error(y.iloc[val_idx], pred)) for pred in meta_preds.T]
    meta_weights = np.array([1/rmse for rmse in meta_rmses])
    meta_weights = meta_weights / meta_weights.sum()
    
    stacking_oof[val_idx] = np.average(meta_preds, axis=1, weights=meta_weights)
    
    # Scale test predictions for linear models
    all_test_preds_scaled = scaler.transform(all_test_preds)
    test_meta_preds = np.column_stack([
        ridge.predict(all_test_preds_scaled),
        lasso.predict(all_test_preds_scaled),
        elastic.predict(all_test_preds_scaled),
        xgb_meta.predict(all_test_preds),
        lgb_meta.predict(all_test_preds),
        cat_meta.predict(all_test_preds)
    ])
    stacking_test_preds += np.average(test_meta_preds, axis=1, weights=meta_weights) / 4

stacking_rmse = np.sqrt(mean_squared_error(y, stacking_oof))
print(f'Stacking OOF RMSE: {stacking_rmse:.5f}')


Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[329]	valid_0's l2: 75.4106


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[378]	valid_0's l2: 75.5498


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[590]	valid_0's l2: 74.9481


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 120 rounds
Did not meet early stopping. Best iteration is:
[599]	valid_0's l2: 74.3155


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Stacking OOF RMSE: 8.66922


In [9]:
fold_weights_list = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    fold_oof = all_oof[val_idx]
    fold_y = y.iloc[val_idx]
    
    # Use blend of Ridge, Lasso, ElasticNet optimizers
    ridge_opt = Ridge(alpha=0.5, random_state=42)
    lasso_opt = Lasso(alpha=0.1, random_state=42, max_iter=2000)
    elastic_opt = ElasticNet(alpha=0.3, l1_ratio=0.5, random_state=42, max_iter=2000)
    
    ridge_opt.fit(fold_oof, fold_y)
    lasso_opt.fit(fold_oof, fold_y)
    elastic_opt.fit(fold_oof, fold_y)
    
    ridge_pred = ridge_opt.predict(fold_oof)
    lasso_pred = lasso_opt.predict(fold_oof)
    elastic_pred = elastic_opt.predict(fold_oof)
    
    opt_rmses = [
        np.sqrt(mean_squared_error(fold_y, ridge_pred)),
        np.sqrt(mean_squared_error(fold_y, lasso_pred)),
        np.sqrt(mean_squared_error(fold_y, elastic_pred))
    ]
    opt_weights = np.array([1/rmse for rmse in opt_rmses])
    opt_weights = opt_weights / opt_weights.sum()
    
    # Blend the coefficients
    fold_weights = (opt_weights[0] * ridge_opt.coef_ + 
                   opt_weights[1] * lasso_opt.coef_ + 
                   opt_weights[2] * elastic_opt.coef_)
    fold_weights = np.maximum(fold_weights, 0)
    fold_weights = fold_weights / fold_weights.sum()
    
    fold_weights_list.append(fold_weights)

optimized_weights = np.mean(fold_weights_list, axis=0)
optimized_weights = optimized_weights / optimized_weights.sum()

simple_ensemble = np.average(all_test_preds, axis=1, weights=optimized_weights)

ratios = np.arange(0.6, 0.95, 0.01)
best_ratio = 0.8
best_rmse = float('inf')

simple_ensemble_oof = np.average(all_oof, axis=1, weights=optimized_weights)
for ratio in ratios:
    test_oof = ratio * stacking_oof + (1 - ratio) * simple_ensemble_oof
    test_rmse = np.sqrt(mean_squared_error(y, test_oof))
    if test_rmse < best_rmse:
        best_rmse = test_rmse
        best_ratio = ratio

print(f'Best stacking ratio: {best_ratio:.2f} (RMSE: {best_rmse:.5f})')

final_predictions = best_ratio * stacking_test_preds + (1 - best_ratio) * simple_ensemble

submission = pd.DataFrame({
    'id': test_ids,
    'exam_score': final_predictions
})
submission.to_csv('submission.csv', index=False)


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.357e+05, tolerance: 6.005e+03
  model = cd_fast.enet_coordinate_descent(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.011e+05, tolerance: 6.005e+03
  model = cd_fast.enet_coordinate_descent(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the

Best stacking ratio: 0.94 (RMSE: 8.66992)


In [10]:
final_oof = best_ratio * stacking_oof + (1 - best_ratio) * simple_ensemble_oof
final_rmse = np.sqrt(mean_squared_error(y, final_oof))
print(f'Final Ensemble OOF RMSE: {final_rmse:.5f}')


Final Ensemble OOF RMSE: 8.66992


In [11]:
import winsound
import time
notes = [(523, 200), (659, 200), (784, 200), (1047, 400), (784, 200), (1047, 600)]
for freq, dur in notes:
    winsound.Beep(freq, dur)
    time.sleep(0.05)
